# ML Week 19: Autoencoders



In [1]:
import torch

In [9]:
from torch import nn

input = torch.arange(1, 5, dtype=torch.float32).view(1, 1, 2, 2)

input

tensor([[[[1., 2.],
          [3., 4.]]]])

In [10]:
m = nn.Upsample(scale_factor=2, mode='nearest')
m(input)

tensor([[[[1., 1., 2., 2.],
          [1., 1., 2., 2.],
          [3., 3., 4., 4.],
          [3., 3., 4., 4.]]]])

In [11]:
m = nn.Upsample(scale_factor=2, mode='bilinear')
m(input)

tensor([[[[1.0000, 1.2500, 1.7500, 2.0000],
          [1.5000, 1.7500, 2.2500, 2.5000],
          [2.5000, 2.7500, 3.2500, 3.5000],
          [3.0000, 3.2500, 3.7500, 4.0000]]]])

In [18]:
%pip install --upgrade huggingface_hub transformers datasets

   ---------------------------------------- 0.0/684.4 kB ? eta -:--:--
   ---------------------------------------- 684.4/684.4 kB 9.1 MB/s  0:00:00
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 8.3 MB/s  0:00:00
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   --- ------------------------------------ 2.1/27.3 MB 12.0 MB/s eta 0:00:03
   ------- -------------------------------- 5.2/27.3 MB 13.6 MB/s eta 0:00:02
   ------------ --------------------------- 8.7/27.3 MB 14.4 MB/s eta 0:00:02
   ------------------ --------------------- 12.3/27.3 MB 15.1 MB/s eta 0:00:01
   ---------------------- ----------------- 15.5/27.3 MB 15.2 MB/s eta 0:00:01
   ---------------------------- ----------- 19.1/27.3 MB 15.5 MB/s eta 0:00:01
   -------------------------------- ------- 22.3/27.3 MB 15.7 MB/s eta 0:00:01
   ------------------------------------ --- 25.2/27.3 MB 15.5 MB/s eta 0:00:01
   --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [20]:
from huggingface_hub import HfApi

In [1]:
%pip install safetensors

Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import torch
from PIL import Image

from transformers import ViTImageProcessor, ViTMAEForPreTraining


In [2]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from transformers import ViTMAEModel

model = ViTMAEModel.from_pretrained(
    "facebook/vit-mae-base"
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTMAEModel LOAD REPORT from: facebook/vit-mae-base
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_before.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.layernorm_before.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc2.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.q_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.mlp.fc1.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.k_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.v_proj.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3, 4, 5, 6, 7}.attention.v_proj.bias   | UNEXPECTED |  | 
decoder.decoder_layers.

In [4]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

processor = ViTImageProcessor.from_pretrained("facebook/vit-mae-base")
inputs = processor(image, return_tensors="pt").to("cpu")
inputs = {k: v.to("cpu") for k, v in inputs.items()}

model = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base", attn_implementation="sdpa", device_map="cpu")
with torch.no_grad():
    outputs = model(**inputs)

reconstruction = outputs.logits

Loading weights:   0%|          | 0/334 [00:00<?, ?it/s]